## How to run in Colab
1. Open [Google Colab](https://colab.research.google.com) → **File → Upload notebook** → choose this file.
2. Download the dataset first: click [Data Science Job Salary dataset.zip](https://github.com/devtlv/Datasets-DA-Bootcamp-2-/raw/refs/heads/main/Week%204%20-%20Data%20Understanding/W4D4%20-%20Data%20Preprocessing%20&%20T/Data%20Science%20Job%20Salary%20dataset.zip). It downloads `Data Science Job Salary dataset.zip`. Save it in `C:\Users\sivan\Datasets`.
3. When the upload cell asks for a file, choose `Data Science Job Salary dataset.zip` from `C:\Users\sivan\Datasets`.

**Run the cells from top to bottom** (Shift+Enter runs one cell). After a restart, run them again from the top.

# Daily Challenge: Data Handling and Analysis in Python

## What You Will Learn
- Advanced techniques for data normalization, reduction, and aggregation.
- Skills in gathering, exploring, integrating, and cleaning data using Python.
- Proficiency in using Pandas for complex data manipulation.

## Your Task
- Download and import the [Data Science Job Salary dataset](https://github.com/devtlv/Datasets-DA-Bootcamp-2-/raw/refs/heads/main/Week%204%20-%20Data%20Understanding/W4D4%20-%20Data%20Preprocessing%20&%20T/Data%20Science%20Job%20Salary%20dataset.zip).
- Normalize the ‘salary’ column using Min-Max normalization which scales all salary values between 0 and 1.
- Implement dimensionality reduction like Principal Component Analysis (PCA) or t-SNE to reduce the number of features (columns) in the dataset.
- Group the dataset by the ‘experience_level’ column and calculate the average and median salary for each experience level (e.g., Junior, Mid-level, Senior).

**Hint :**
- As a reminder, normalization is crucial when dealing with data that has different ranges. For example, salary data might have a wide range (e.g., from \$20,000 to \$200,000). By scaling the data using Min-Max normalization, you make sure that all salary values fall within a consistent range (0 to 1). This is particularly helpful when the data is going to be used in machine learning models, as some algorithms (like k-nearest neighbors or neural networks) perform better when features are normalized. It ensures that no single salary dominates the learning process, making the analysis more balanced.
- Dimensionality reduction helps simplify complex datasets by reducing the number of variables under consideration. This can make the data more manageable and help avoid the curse of dimensionality—a phenomenon where machine learning models struggle when dealing with high-dimensional data. PCA, for instance, helps in retaining the most important information (variance) from the dataset while reducing noise and redundancy. It can also speed up the training process for models and help in visualizing data in fewer dimensions.
- Aggregating data helps in understanding trends within subgroups of the dataset. Calculating average and median salaries for each experience level gives insights into the compensation distribution and disparities across different job levels. This kind of aggregation can help in answering business questions like “How does salary evolve with experience?” or “What is the salary distribution for senior-level roles?”

## Submit your Daily Challenge
Upload your Python scripts, visualizations, and a summary of your insights on GitHub.

## Setup
Run this cell first. It imports the libraries used in the challenge.

In [ ]:
import pandas as pd
# pandas: for working with tables (DataFrames)
import matplotlib.pyplot as plt
# matplotlib: for drawing charts
import seaborn as sns
# seaborn: for nicer statistical charts (built on matplotlib)

---
# Step 1: Load and look at the data

In [ ]:
from google.colab import files
# the Colab tool for uploading files from your computer
uploaded = files.upload()
# choose "Data Science Job Salary dataset.zip" (saved in a variable so its contents are not printed)
!unzip "Data Science Job Salary dataset.zip"
# unzip it: you get the folder "Data Science Job Salary dataset" with "datascience_salaries.csv"

In [ ]:
df = pd.read_csv('Data Science Job Salary dataset/datascience_salaries.csv')
# load the CSV file into a DataFrame called df
print(df.shape)
# (number of rows, number of columns)
print(df.head())
# the first 5 rows

In [ ]:
df.info()
# the column names, their types and how many values are not empty

**What we see:** the print shows **1171 rows and 7 columns**. The salary column is called **`salary`** (there is no `salary_in_usd` column), and it is a number (int64). `Unnamed: 0` is only a saved row number, so we drop it. `df.info()` shows **1171 non-null** values in every column, so nothing is empty.

In [ ]:
df = df.drop(columns=['Unnamed: 0'])
# remove the extra index column (it is only a row number saved in the file)
print(df.columns.tolist())
# the columns we keep

### Check missing values and duplicates

In [ ]:
print(df.isnull().sum())
# count empty cells in each column
print('Duplicate rows:', df.duplicated().sum())
# count rows that are exact copies of an earlier row

**Missing values and duplicates:** the print shows **0** missing values in every column. It also shows **345** rows that are exact copies of an earlier row. We **keep** them: the file has no job ID, so two rows like "Data scientist, Senior, New York City, 149000" (rows 0 and 4 in the `head()` print above) can be two real job posts with the same details. Removing them would change how many jobs each experience level has.

### Check the currency of the salaries

In [ ]:
print(df['salary_currency'].value_counts())
# how many salaries are in each currency

**Currency:** the print shows **1157 salaries in USD, 9 in EUR and 5 in GBP**. A salary of 60000 EUR and 60000 USD are not the same money, and the file has no USD column for them, so we keep only the **USD** rows. The next print shows **1157** rows left (we lose only 14 rows).

In [ ]:
df = df[df['salary_currency'] == 'USD']
# keep only the rows where the salary is in US dollars
print('Rows left:', len(df))
# how many rows are left

---
# Step 2: Normalize the salary column with Min-Max

In [ ]:
print(df['salary'].describe())
# the salary before scaling: min, max, mean, quartiles

In [ ]:
from sklearn.preprocessing import MinMaxScaler
# the scaler that turns numbers into the range 0 to 1
scaler = MinMaxScaler()
# create the scaler
df['salary_normalized'] = scaler.fit_transform(df[['salary']])
# fit learns the min and max salary, transform turns each salary into (salary - min) / (max - min)
print(df[['salary', 'salary_normalized']].head())
# compare the real and normalized salary
print('Min:', df['salary_normalized'].min(), '| Max:', df['salary_normalized'].max())
# the normalized column goes from 0 to 1

**Explanation:** the `describe()` print shows the salary goes from **30000 to 228000** USD. Min-Max scaling turns this into **0 to 1** (the last print shows Min 0.0 and Max 1.0): the lowest salary becomes 0, the highest becomes 1, and for example 149000 becomes **0.60**. We keep the real `salary` too, for the groupby in Step 4.

---
# Step 3: Dimensionality reduction (PCA) on the encoded features

PCA works only on numbers, so first we encode the text columns with one-hot encoding (`get_dummies`). We look at how many different values each text column has, because each value becomes a new column.

In [ ]:
print(df[['job_title', 'job_type', 'experience_level', 'location']].nunique())
# how many different values each text column has

**Choice of columns:** the print shows `job_title` has **5** values, `job_type` **2** and `experience_level` **4**, but `location` has **318** different values. One-hot encoding location would add 318 columns for only 1157 rows, so we leave location out and encode the other 3. With the normalized salary, the encoded table has **12 columns** (the shape print shows (1157, 12)).

In [ ]:
features = df[['job_title', 'job_type', 'experience_level', 'salary_normalized']]
# the columns we give to PCA (location is left out, see the explanation above)
encoded = pd.get_dummies(features, columns=['job_title', 'job_type', 'experience_level'], dtype=int)
# one-hot encode the 3 text columns: one 0/1 column for each value
print(encoded.shape)
# rows and columns after encoding
print(encoded.columns.tolist())
# the names of the encoded columns

In [ ]:
from sklearn.decomposition import PCA
# PCA = Principal Component Analysis, it squeezes many columns into a few new ones
pca = PCA(n_components=2)
# keep 2 components so we can draw them on a flat chart
components = pca.fit_transform(encoded)
# fit finds the 2 directions with the most variation, transform gives each row its 2 new values
print('Variance explained by each component:', pca.explained_variance_ratio_.round(3))
# the share of the total variation each component keeps
print('Total variance kept:', round(pca.explained_variance_ratio_.sum(), 3))
# the share both components keep together

In [ ]:
loadings = pd.DataFrame(pca.components_.T, index=encoded.columns, columns=['PC1', 'PC2'])
# how much each encoded column counts in each component (big + or - number = important)
print(loadings.round(2))
# show the table

In [ ]:
pca_df = pd.DataFrame(components, columns=['PC1', 'PC2'])
# put the 2 components in a small table
pca_df['experience_level'] = df['experience_level'].values
# add the experience level of each row, to color the points
sns.scatterplot(data=pca_df, x='PC1', y='PC2', hue='experience_level', alpha=0.6)
# draw each job as a point, colored by experience level
plt.title('PCA of the encoded job features')
# chart title
plt.show()
# show the chart

**Explanation:** PCA squeezed the 12 encoded columns into 2. The print shows PC1 keeps **29.4%** and PC2 **23.9%** of the variation, **53.3%** together. The loadings table shows what each component stands for:
- **PC1** is mostly the experience level: **Senior +0.73** against **Mid -0.58**.
- **PC2** is mostly the job title: **Data scientist +0.74** against **Data analyst -0.59**.

That is why the scatter plot shows separate small clusters: the blue Senior points are on the right (high PC1) and the green Mid points on the left. Each cluster is one mix of job title and level, because the features are 0/1 columns, so many jobs get exactly the same point.

---
# Step 4: Average and median salary by experience level

In [ ]:
salary_by_level = df.groupby('experience_level')['salary'].agg(['mean', 'median', 'count'])
# for each experience level: the average salary, the median salary and the number of jobs
salary_by_level = salary_by_level.sort_values('median')
# sort from the lowest to the highest median
print(salary_by_level.round(0))
# show the table (rounded to whole dollars)

In [ ]:
salary_by_level[['mean', 'median']].plot(kind='bar')
# bar chart: the mean and median salary side by side for each level
plt.title('Salary by experience level (USD)')
# chart title
plt.ylabel('Salary')
# label of the y axis
plt.xticks(rotation=0)
# keep the level names horizontal
plt.show()
# show the chart

**Explanation:** the exercise gives the levels as examples (Junior, Mid-level, Senior). In this file the `experience_level` values are **Entry, Mid, Senior and Executive** (Entry is the junior level). The table shows the mean and median salary (USD) for each level, sorted by the median:
- **Entry:** mean **36111**, median **30000** (126 jobs)
- **Executive:** mean **76077**, median **46000** (13 jobs)
- **Mid:** mean **51813**, median **51000** (299 jobs)
- **Senior:** mean **75403**, median **68000** (719 jobs)

For Entry, Mid and Senior the salary goes up with experience. **Executive** is strange: its mean (76077) is much higher than its median (46000). It has only **13** jobs, so a few very high salaries pull the mean up, while the middle Executive salary is lower than the Mid median.

---
# Summary of insights

1. **The data:** 1171 jobs, no missing values; we kept the **1157 USD** salaries so all salaries are in the same money. The salary column is `salary` (there is no `salary_in_usd`).
2. **Normalization:** Min-Max turned salaries from 30000–228000 USD into 0–1, so salary has the same range as the 0/1 encoded columns before PCA.
3. **PCA:** 2 components keep **53.3%** of the variation. PC1 separates **Senior from Mid** jobs and PC2 separates **Data scientist from Data analyst** jobs, so experience level and job title are what make the jobs most different.
4. **Salary by experience:** the median rises from **30000 (Entry)** to **51000 (Mid)** to **68000 (Senior)**: more experience means a higher salary.
5. **Mean vs median:** for Executive the mean is **76077** but the median only **46000**, with just **13** jobs. With so few jobs, a few high salaries move the mean a lot, so the median is the safer number to compare levels.